# 06 — Final Fit and Submission

**Objective.** Refit the selected architecture on all 500 calibration queries, predict all test queries, validate the submission contract, and write `answer.csv`.

Run this notebook only after notebook 05 has frozen the architecture and parameters.


## Setup


In [ ]:
from pathlib import Path
import os, sys, json, time, copy, itertools, importlib.util, subprocess

def locate_project(start=Path.cwd()):
    for parent in [start.resolve(), *start.resolve().parents]:
        if (parent / "pyproject.toml").exists() and (parent / "src/avito_retriever").exists():
            return parent
    candidate = Path("/content/avito-retriever")
    if candidate.exists():
        return candidate
    raise FileNotFoundError("Clone/open avito-retriever or change PROJECT_ROOT in this cell")

PROJECT_ROOT = locate_project()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

def module_available(name):
    try:
        return importlib.util.find_spec(name) is not None
    except ModuleNotFoundError:
        return False

IN_COLAB = module_available("google.colab")
if IN_COLAB and os.environ.get("AVITO_AUTO_INSTALL", "1") != "0":
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "-e",
        f"{PROJECT_ROOT}[lexical,neural,dev]",
    ])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from avito_retriever.experiments.notebook import resolve_data_dir, result_dir, highlight_best, save_json, save_yaml

DATA_DIR = resolve_data_dir(PROJECT_ROOT)
SEED = 42
np.random.seed(SEED)
print("Project:", PROJECT_ROOT)
print("Data:", DATA_DIR)


In [ ]:
from avito_retriever.data.io import load_articles, load_calibration, load_test
from avito_retriever.cli.validate_submission import validate_submission_frame

R=PROJECT_ROOT/"artifacts/notebook_results"
decision_path=R/"05_final_architecture_significance/final_selected_architecture.json"
assert decision_path.exists(), "Run notebook 05 first"
decision=json.loads(decision_path.read_text())
articles=load_articles(DATA_DIR); calibration=load_calibration(DATA_DIR); test=load_test(DATA_DIR)
display(pd.DataFrame([decision]))


## Final prediction runner

The production runner uses the same cached artifacts and selected YAML/JSON parameters as notebooks 01–04. It must not read test labels or hardcode query IDs.


In [ ]:
from avito_retriever.pipeline.final import fit_predict_selected

rankings, run_manifest = fit_predict_selected(
    project_root=PROJECT_ROOT,
    data_dir=DATA_DIR,
    decision=decision,
    top_k=10,
)
display(pd.DataFrame([run_manifest]))
display(rankings.head(20))


## Build and validate answer.csv


In [ ]:
answers=(rankings.sort_values(["query_id","rank"]).groupby("query_id").article_id
         .apply(lambda values:" ".join(map(str,list(dict.fromkeys(values))[:10]))).rename("answer").reset_index())
answers=test[["query_id"]].merge(answers,on="query_id",how="left")
validate_submission_frame(answers,test,set(articles.article_id.astype(int)),max_k=10)
submission_dir=PROJECT_ROOT/"artifacts/submissions"; submission_dir.mkdir(parents=True,exist_ok=True)
answer_path=submission_dir/"answer.csv"; answers.to_csv(answer_path,index=False)
print("Valid answer.csv:",answer_path,"rows=",len(answers))
display(answers.head(10))


## Reproducibility manifest


In [ ]:
save_json({**run_manifest,"answer_csv":str(answer_path),"rows":len(answers),
           "max_articles":int(answers.answer.str.split().str.len().max())}, submission_dir/"submission_manifest.json")


## Create a shareable analysis bundle


In [ ]:
from avito_retriever.experiments.bundle import build_analysis_bundle
bundle_path, bundle_manifest = build_analysis_bundle(PROJECT_ROOT)
print(f"Send this file for analysis: {bundle_path}")
print(f"Bundle size: {bundle_manifest['bundle_bytes']/1024**2:.2f} MB; files: {bundle_manifest['file_count']}")


## Expected output

- `artifacts/submissions/answer.csv` — file to submit;
- `submission_manifest.json` — chosen architecture, models, parameters and artifact paths;
- `artifacts/analysis_bundles/avito-results-*.zip` — compact package to send for analysis;
- a validated preview with exactly 500 rows and no invalid article IDs.
